In [159]:
import pandas as pd              # 표 형태 데이터 처리 라이브러리
import OpenDartReader            # DART API 라이브러리
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import contextlib
import io
import FinanceDataReader as fdr


In [160]:
api_key = "17b6497bf3f2fcec2d6bd23bebb7d38291813272"     # DART에서 발급받은 API 키
dart = OpenDartReader(api_key)   # API 연결

In [161]:
def get_earnings(corp_name): 
    # 기업명으로 DART 기업코드 찾기
    corp_info = dart.find_corp_code(corp_name)
    
    # 기업이 없으면 종료
    if not corp_info:
        print(f"'{corp_name}'에 해당하는 기업을 찾을 수 없습니다.")
        return None
    
    corp_code = corp_info
    data = []   
    revenue_accounts = ['매출액','영업수익','수익(매출액)']

    for year in range(2015, 2026):

        # try:

        #     with contextlib.redirect_stdout(io.StringIO()):
        #         df_shares = dart.report(corp_code,'주식총수',year)
        #         common = df_shares[df_shares['se'] == '보통주']
        #         shares = int(str(common['istc_totqy'].values[0].replace(',',''))) # 발행주식수

        # except: 
        #     shares = None
        
        with contextlib.redirect_stdout(io.StringIO()):
            df = dart.finstate(corp_code, year)

        revenue = None
        operating_profit = None
        net_income = None

        if df is not None and not df.empty:

            op_row = df[df['account_nm'] == '영업이익']
            if not op_row.empty:
                operating_profit = op_row['thstrm_amount'].values

            net_row = df[df['account_nm'] == '당기순이익(손실)']
            if not op_row.empty:
                net_income = net_row['thstrm_amount'].values

            # 매출 계정 찾기
            rev_row = df[df['account_nm'].isin(revenue_accounts)]

            # finstate에 매출이 없으면 finstate_all에서 찾기
            if rev_row.empty and 'reprt_code' in df.columns:
                with contextlib.redirect_stdout(io.StringIO()):
                    df_all = dart.finstate_all(corp_code, year)

                if df_all is not None and not df_all.empty:
                    rev_row = df_all[df_all['account_nm'].isin(revenue_accounts)]

            if not rev_row.empty:
                revenue = rev_row['thstrm_amount'].values

        data.append({
            "연도": year,
            "매출액": int(str(revenue[0]).replace(',', '')) if revenue is not None and len(revenue) > 0 else None,
            "영업이익": int(str(operating_profit[0]).replace(',', '')) if operating_profit is not None and len(operating_profit) > 0 else None,
            "당기순이익": int(str(net_income[0]).replace(',', '')) if net_income is not None and len(net_income) > 0 else None
        })

    result = pd.DataFrame(data)
    result['영업이익률'] = ((result['영업이익'] / result['매출액']) * 100).round(2)
    result = result.set_index('연도')

    return result

In [162]:
def get_earnings_p(corp_name):

    df = get_earnings(corp_name)

    display(
        df.style.format({
            "매출액": "{:,.0f}",
            "영업이익": "{:,.0f}",
            "영업이익률": "{:.2f}",
            "당기순이익": "{:,.0f}",
        })
    )

In [163]:
def get_earnings_graph(corp_name):

    df_result = get_earnings(corp_name)
    
    # 데이터가 비어있는지 확인
    if df_result.empty:
        print("❌ 수집된 데이터가 없습니다. 기업 코드를 확인해주세요.")
        return

    plt.rc('font', family='Malgun Gothic')

    # [데이터 전처리] 단위 변경
    revenue = df_result['매출액'] / 1e8
    op_profit = df_result['영업이익'] / 1e8

    fig, ax1 = plt.subplots(figsize=(12, 6)) 
    plt.xticks(df_result.index)

    # 왼쪽 축: 매출액 막대
    ax1.bar(df_result.index, revenue, color='dodgerblue', alpha=0.7, label='매출액', width=0.6)
    ax1.set_title(corp_name , fontsize=16)
    ax1.set_ylabel('매출액 (억원)', color='dodgerblue')
    ax1.tick_params(axis='y', labelcolor='dodgerblue')
    ax1.yaxis.set_major_formatter(ticker.StrMethodFormatter('{x:,.0f}'))

    # 오른쪽 축: 영업이익 꺾은선
    ax2 = ax1.twinx() 
    ax2.plot(df_result.index, op_profit, color='orangered', marker='o', linewidth=2, label='영업이익')
    ax2.set_ylabel('영업이익 (억원)', color='orangered')
    ax2.tick_params(axis='y', labelcolor='orangered')
    ax2.yaxis.set_major_formatter(ticker.StrMethodFormatter('{x:,.0f}'))

    # 범례 합치기
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

    ax1.grid(axis='y', linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show() # ✅ 호출될 때만 그래프를 띄움

In [164]:
# def get_per(corp_name):
#     # 1. 일꾼(get_earnings)을 시켜서 기본 재무 데이터를 가져옵니다.
#     # (주의: get_earnings 마지막의 .map(lambda...) 포맷팅 로직은 제거된 상태여야 함)
#     df = get_earnings(corp_name)
    
#     if df is None or df.empty:
#         return None
    
#     # 2. 기업 코드를 다시 확보 (주가 조회를 위해)
#     corp_code = dart.find_corp_code(corp_name)
    
#     # 3. 주가 수집 및 PER 계산
#     prices = []
#     for year in df.index:
#         try:
#             # FinanceDataReader로 각 연도 12월 주가 조회
#             price_df = fdr.DataReader(corp_code, f"{year}-12-01", f"{year}-12-31") #dart 코드가 아닌 주식 티커 필요
#             price = price_df['Close'].iloc[-1] if not price_df.empty else None
#         except:
#             price = None
#         prices.append(price)
    
#     df['기말주가'] = prices
    
#     # 4. PER 계산 (PER = 주가 / EPS)
#     # EPS(주당순이익)가 데이터에 있다고 가정하거나, 당기순이익/주식수로 계산
#     # 여기서는 예시로 '당기순이익'과 '주식수'가 df에 포함되어 있다고 가정합니다.
#     if '당기순이익' in df.columns and '주식수' in df.columns:
#         df['EPS'] = (df['당기순이익'] / df['주식수'])
#         df['PER'] = (df['기말주가'] / df['EPS']).round(2)

#     # 5. 모든 계산이 끝난 후 마지막에 보기 좋게 포맷팅 (콤마 찍기)
#     display_df = df.map(lambda x: f"{x:,.2f}" if pd.notnull(x) and isinstance(x, (int, float)) else x)
    
#     return display_df

In [166]:
# def get_codes(corp_name):
#     corp_list = dart.corp_codes
#     df = corp_list[corp_list['corp_name'] == corp_name]
    
#     if not df.empty:
#         return df.iloc[0]['stock_code']
#     else:
#         return None


# stock_code = get_codes('삼성전자')

# print(stock_code)

In [168]:
get_earnings_p('삼성전자')

,매출액,영업이익,당기순이익,영업이익률
연도,,,,
2015,"200,653,482,000,000","26,413,442,000,000","19,060,144,000,000",13.16
2016,"201,866,745,000,000","29,240,672,000,000","22,726,092,000,000",14.49
2017,"239,575,376,000,000","53,645,038,000,000","42,186,747,000,000",22.39
2018,"243,771,415,000,000","58,886,669,000,000","44,344,857,000,000",24.16
2019,"230,400,881,000,000","27,768,509,000,000","21,738,865,000,000",12.05
2020,"236,806,988,000,000","35,993,876,000,000","26,407,832,000,000",15.20
2021,"279,604,799,000,000","51,633,856,000,000","39,907,450,000,000",18.47
2022,"302,231,360,000,000","43,376,630,000,000","55,654,077,000,000",14.35
2023,"258,935,494,000,000","6,566,976,000,000","15,487,100,000,000",2.54
